# SWAMPE-JAX (`my_swamp`) — vmap GPU throughput sweep

`jax.vmap` runs a **batch of trajectories** (a sweep of `DPhieq`) in one compiled call. The point
of this notebook is to find the batch size that uses the GPU **efficiently**, not the biggest that
merely fits in memory.

**Method:** **10-day** runs (matches the CPU speed benchmark window), **doubling the batch**
`N = 1, 2, 4, 8, …` and measuring trajectories/second each time. While the GPU is underutilized,
doubling `N` barely changes the batch time, so throughput climbs. Once the GPU saturates
(compute-bound), batch time scales with `N` and **throughput plateaus** — that knee is the
efficient batch size. (Memory would let you go far past this, but those huge batches run at the
same per-trajectory cost, just slower per call.)

**Run:** `Runtime → Change runtime type → GPU (A100)` → `Run all`. Copy the `===== RESULTS =====`
block back to me.

### 1. Install (not timed)  —  `--no-deps` protects Colab's CUDA JAX

In [1]:
!pip install -q --no-deps --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ my-swamp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.3/63.3 kB 6.9 MB/s eta 0:00:00


### 2. Knobs + setup
`CONFIG` matches the CPU/GPU timing tests. The batch varies `DPhieq` across members so the
trajectories are genuinely different runs.

In [ ]:
# ===== knobs =====
PRECISION    = "float64"   # "float64" (matches the CPU test) | "float32" (needs runtime restart)
SWEEP_DAYS   = 10          # run length for every throughput measurement [days] -- matches the CPU speed test window
REPEATS      = 3           # timed runs averaged at each batch size
MAX_BATCH    = 8192        # hard cap / safety stop for the doubling
PLATEAU_TOL  = 0.10        # stop doubling once a doubling adds < 10% throughput
MIN_N        = 4           # don't declare a plateau before reaching this N

import os
USE_X64 = (PRECISION.lower() == "float64")
os.environ["SWAMPE_JAX_ENABLE_X64"] = "1" if USE_X64 else "0"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")  # grow mem on demand

import statistics, subprocess, time
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", USE_X64)

try:
    from my_swamp.model import run_model_scan_final as _my_final
    def my_run(**kw): return _my_final(**kw)
except Exception:
    from my_swamp.model import run_model_scan as _my_scan
    def my_run(**kw): return _my_scan(return_history=False, **kw)

CONFIG = dict(
    M=42, dt=120.0, Phibar=3.0e5, omega=3.2e-5, a=8.2e7, g=9.8,
    test=None, forcflag=True, taurad=10.0*3600.0, taudrag=6.0*3600.0,
    DPhieq=1.0e6, diffflag=True, modalflag=True, alpha=0.01, expflag=False, K6=1.24e33,
)
BASE = {k: v for k, v in CONFIG.items() if k != "DPhieq"}

def tmax_for_days(days, dt):
    n = int(round(days * 86400.0 / dt)); return max(3, n + 1), n

def timeit(fn, warmups, repeats):
    for _ in range(warmups): fn()
    ts = []
    for _ in range(repeats):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return {"mean": statistics.mean(ts), "median": statistics.median(ts), "min": min(ts),
            "std": statistics.pstdev(ts) if len(ts) > 1 else 0.0, "raw": ts}

def gpu_name():
    try:
        o = subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"], capture_output=True, text=True, timeout=10)
        return o.stdout.strip().splitlines()[0] if o.stdout.strip() else "unknown"
    except Exception:
        return "unknown"

GPU = next((d for d in jax.devices() if d.platform == "gpu"), None)

def mem_peak_mb():
    try:
        s = GPU.memory_stats() or {}
        p = s.get("peak_bytes_in_use")
        return None if p is None else p / 1e6
    except Exception:
        return None

def dphieq_batch(n):
    return jnp.linspace(0.5e6, 2.0e6, n)            # dtype follows session precision

def make_batched(tmax):
    def one(dpe):
        return my_run(tmax=tmax, DPhieq=dpe, jit_scan=True, diagnostics=False, **BASE)["last_state"].Phi_curr
    return jax.jit(jax.vmap(one))

def is_oom(e):
    s = str(e).upper()
    return ("RESOURCE_EXHAUSTED" in s) or ("OUT OF MEMORY" in s) or ("OOM" in s)

def run_batched(f, xb):
    with jax.default_device(GPU):
        o = f(xb); o.block_until_ready(); return o

# build_static does host-side numpy on the Legendre basis, so it must run CONCRETELY once
# (under jax.vmap it would np.asarray a tracer). One plain call populates an lru_cache the
# vmap'd version reuses.
my_run(tmax=8, **CONFIG)["last_state"].Phi_curr.block_until_ready()

print(f"jax {jax.__version__}  precision={PRECISION}  x64={jax.config.read('jax_enable_x64')}")
print(f"devices = {jax.devices()}")
print(f"GPU = {gpu_name() if GPU else 'NONE — set an A100 runtime and Run all'}")

### 3. Throughput sweep — double the batch until throughput plateaus
Each row is a 10-day batched run. `thr` is trajectories/second; watch it climb, then flatten.

In [3]:
assert GPU is not None, "No GPU — select an A100 runtime and Run all."

dt = CONFIG["dt"]
tmax, _ = tmax_for_days(SWEEP_DAYS, dt)
steps = tmax - 2
print(f"sweep: M={CONFIG['M']} dt={dt:.0f}s  {SWEEP_DAYS}-day runs ({steps} steps), doubling N up to {MAX_BATCH}")
print(f"{'N':>6} {'batch_s':>9} {'traj/s':>10} {'ms/traj':>9} {'ms/step/traj':>13} {'thr gain':>9} {'mem MB':>9}")

rows = []
N, prev_thr = 1, None
while N <= MAX_BATCH:
    f = make_batched(tmax); xb = dphieq_batch(N)
    try:
        run_batched(f, xb)                                   # compile (excluded from timing)
        st = timeit(lambda: run_batched(f, xb), warmups=0, repeats=REPEATS)
    except Exception as e:
        if is_oom(e):
            print(f"  N={N}: OOM — stopping (well past the efficient point anyway)"); jax.clear_caches(); break
        raise
    T = st["mean"]; thr = N / T; ptt = T / N * 1e3; pspt = T / N / steps * 1e3
    gain = (thr / prev_thr - 1.0) if prev_thr is not None else None
    pk = mem_peak_mb()
    rows.append(dict(N=N, T=T, thr=thr, ptt=ptt, pspt=pspt, std=st["std"], gain=gain, mem=pk))
    print(f"{N:6d} {T:9.4f} {thr:10.1f} {ptt:9.3f} {pspt:13.5f} "
          f"{'   —' if gain is None else f'{gain*100:7.1f}%'} {('—' if pk is None else f'{pk:,.0f}'):>9}")
    if gain is not None and gain < PLATEAU_TOL and N >= MIN_N:
        print(f"  -> throughput plateaued (last doubling added only {gain*100:.1f}% < {PLATEAU_TOL*100:.0f}%)")
        break
    prev_thr = thr; N *= 2

peak = max(rows, key=lambda r: r["thr"])
knee = next(r for r in rows if r["thr"] >= 0.90 * peak["thr"])     # smallest N within 10% of peak
print(f"\npeak throughput : N={peak['N']}  {peak['thr']:,.1f} traj/s  ({peak['ptt']:.3f} ms/traj)")
print(f"efficient knee  : N={knee['N']}  {knee['thr']:,.1f} traj/s  ({knee['ptt']:.3f} ms/traj)  "
      f"<- smallest batch within 10% of peak")

sweep: M=42 dt=120s  1-day runs (719 steps), doubling N up to 8192
     N   batch_s     traj/s   ms/traj  ms/step/traj  thr gain    mem MB
     1    0.1248        8.0   124.850       0.17364    —        76
     2    0.1328       15.1    66.386       0.09233    88.1%        88
     4    0.1522       26.3    38.050       0.05292    74.5%        91
     8    0.1754       45.6    21.926       0.03049    73.5%        91
    16    0.2387       67.0    14.921       0.02075    46.9%        92
    32    0.4136       77.4    12.926       0.01798    15.4%       141
    64    0.7889       81.1    12.326       0.01714     4.9%       163
  -> throughput plateaued (last doubling added only 4.9% < 10%)

peak throughput : N=64  81.1 traj/s  (12.326 ms/traj)
efficient knee  : N=32  77.4 traj/s  (12.926 ms/traj)  <- smallest batch within 10% of peak


### 4. Diagnostics at the efficient batch + results

In [4]:
EFF = knee                      # the recommended (efficient) batch
Nf = EFF["N"]
print(f"detailed run at efficient batch N={Nf} ({SWEEP_DAYS}-day, {steps} steps)\n")

f = make_batched(tmax); xb = dphieq_batch(Nf)
run_batched(f, xb)                                            # ensure compiled
st = timeit(lambda: run_batched(f, xb), warmups=0, repeats=max(REPEATS, 5))
out = run_batched(f, xb)
arr = np.asarray(out)                                         # (Nf, J, I)

single = rows[0]                                              # N=1 row
T = st["mean"]
thr = Nf / T
ptt = T / Nf * 1e3
pspt = T / Nf / steps * 1e3
vmap_eff = single["ptt"] / ptt                               # per-traj speedup vs N=1
finite_frac = float(np.isfinite(arr).mean())
dphi = np.asarray(xb)
per_member_mean = arr.reshape(arr.shape[0], -1).mean(axis=1)
pk = mem_peak_mb()

print("--- SWEEP TABLE ------------------------------------------------")
print(f"{'N':>6} {'batch_s':>9} {'traj/s':>10} {'ms/traj':>9} {'thr gain':>9}")
for r in rows:
    g = '   —' if r['gain'] is None else f"{r['gain']*100:7.1f}%"
    print(f"{r['N']:6d} {r['T']:9.4f} {r['thr']:10.1f} {r['ptt']:9.3f} {g:>9}")

print("\n--- DIAGNOSTICS AT EFFICIENT BATCH -----------------------------")
print(f"gpu                       : {gpu_name()}")
print(f"precision                 : {PRECISION}")
print(f"efficient batch N         : {Nf}   (peak-throughput N = {peak['N']})")
print(f"run length                : {SWEEP_DAYS} day = {steps} steps")
print(f"output shape              : {arr.shape}  finite_frac={finite_frac:.6f}")
print(f"per-run batch times [s]   : {[round(t,4) for t in st['raw']]}")
print(f"batch time mean/std/min   : {st['mean']:.4f} / {st['std']:.4f} / {st['min']:.4f} s")
print(f"throughput                : {thr:,.1f} trajectories/s")
print(f"per trajectory            : {ptt:.4f} ms   ({pspt:.5f} ms/step/traj)")
print(f"single-traj (N=1)         : {single['ptt']:.3f} ms/traj  -> vmap is ~{vmap_eff:.1f}x more efficient")
print(f"GPU mem peak              : {('n/a' if pk is None else f'{pk:,.0f} MB')}")
print(f"final Phi global min/mean/max : {arr.min():.6g} / {arr.mean():.6g} / {arr.max():.6g}")
k = min(5, Nf)
print(f"per-member final-Phi mean (first {k}):")
for i in range(k):
    print(f"    DPhieq={float(dphi[i]):.3e} -> mean(Phi)={float(per_member_mean[i]):.6g}")
if Nf > k:
    print(f"per-member final-Phi mean (last {k}):")
    for i in range(Nf - k, Nf):
        print(f"    DPhieq={float(dphi[i]):.3e} -> mean(Phi)={float(per_member_mean[i]):.6g}")

print("\n===== RESULTS (copy everything below back to me) =====")
print(f"precision               = {PRECISION}")
print(f"gpu                     = {gpu_name()}")
print(f"M                       = {CONFIG['M']}")
print(f"dt_seconds              = {dt:.0f}")
print(f"sweep_days              = {SWEEP_DAYS}")
print(f"steps                   = {steps}")
print(f"efficient_batch_N       = {Nf}")
print(f"peak_throughput_N       = {peak['N']}")
print(f"throughput_traj_per_s   = {thr:.1f}")
print(f"peak_throughput_traj_s  = {peak['thr']:.1f}")
print(f"ms_per_trajectory       = {ptt:.4f}")
print(f"ms_per_step_per_traj    = {pspt:.5f}")
print(f"single_traj_ms_per_traj = {single['ptt']:.4f}")
print(f"vmap_efficiency_x       = {vmap_eff:.2f}")
print(f"gpu_mem_peak_MB         = {('NA' if pk is None else round(pk))}")
print(f"finite_frac             = {finite_frac:.6f}")
print("sweep_N_thr =", [(r["N"], round(r["thr"], 1)) for r in rows])
print("===== END RESULTS =====")

detailed run at efficient batch N=32 (1-day, 719 steps)

--- SWEEP TABLE ------------------------------------------------
     N   batch_s     traj/s   ms/traj  thr gain
     1    0.1248        8.0   124.850         —
     2    0.1328       15.1    66.386     88.1%
     4    0.1522       26.3    38.050     74.5%
     8    0.1754       45.6    21.926     73.5%
    16    0.2387       67.0    14.921     46.9%
    32    0.4136       77.4    12.926     15.4%
    64    0.7889       81.1    12.326      4.9%

--- DIAGNOSTICS AT EFFICIENT BATCH -----------------------------
gpu                       : NVIDIA A100-SXM4-40GB
precision                 : float64
efficient batch N         : 32   (peak-throughput N = 64)
run length                : 1 day = 719 steps
output shape              : (32, 64, 128)  finite_frac=1.000000
per-run batch times [s]   : [0.4132, 0.4132, 0.4132, 0.4132, 0.4133]
batch time mean/std/min   : 0.4132 / 0.0000 / 0.4132 s
throughput                : 77.4 trajectories/s
pe